In [1]:
import faiss
import json
from FlagEmbedding import FlagModel
from whoosh import index
from whoosh.qparser import QueryParser, OrGroup

c:\Users\hrkwl\.conda\envs\bible_rag\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
model = FlagModel('BAAI/bge-small-en-v1.5', 
                  query_instruction_for_retrieval="Represent this sentence for searching relevant passages: ",
                  use_fp16=True)

In [3]:
faiss_idx = faiss.read_index("../data/processed/bge-small-en-v1.5.faiss")

In [4]:
faiss_idx

<faiss.swigfaiss_avx2.IndexFlatIP; proxy of <Swig Object of type 'faiss::IndexFlatIP *' at 0x000001BECE155DA0> >

In [5]:
instruction = "Represent this sentence for searching relevant passages: "

In [6]:
query = "Are there any verses about love?"

In [7]:
query_embeddings = model.encode_queries([query])

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


In [8]:
query_embeddings.shape

(1, 384)

In [9]:
similarities, indices = faiss_idx.search(query_embeddings, k=50)

In [10]:
similarities

array([[0.7125931 , 0.70533985, 0.6916112 , 0.688118  , 0.6857797 ,
        0.6842478 , 0.6828175 , 0.6740644 , 0.67395765, 0.6726903 ,
        0.6713309 , 0.6701046 , 0.6694894 , 0.6680068 , 0.6676769 ,
        0.6672876 , 0.66664875, 0.6665238 , 0.6661459 , 0.66606987,
        0.6658249 , 0.6656207 , 0.66538274, 0.6652349 , 0.6646593 ,
        0.6639215 , 0.66239935, 0.6621903 , 0.66113245, 0.66091967,
        0.66056454, 0.6601709 , 0.6580935 , 0.65808296, 0.6576822 ,
        0.6575488 , 0.6572114 , 0.65720123, 0.6570623 , 0.65691817,
        0.6550497 , 0.65410644, 0.6539198 , 0.65321475, 0.65275395,
        0.65226305, 0.6520376 , 0.6510001 , 0.65078133, 0.648944  ]],
      dtype=float32)

In [11]:
indices

array([[24025, 24016, 15793, 24004, 24017,  6179, 17761, 23344, 22531,
        24021, 22435, 23199, 14118,  3900, 23854, 23324, 22572, 22434,
        20265, 24053, 24019, 23107, 22354, 15239, 14085, 23963, 23294,
        17057,  3988, 14022, 21343, 21260, 23490, 18347, 14139, 23487,
         6301, 22848, 23512, 15261, 23988, 23338, 14075, 19746, 11259,
        15254, 15431, 22433, 16776, 20141]])

In [12]:
with open('../data/processed/bible.json', encoding="utf-8") as f:
  bible = json.load(f)

In [13]:
bible[16776]

{'text': '21:7 And it shall be, when they say unto thee, Wherefore sighest thou? that thou shalt answer, For the tidings; because it cometh: and every heart shall melt, and all hands shall be feeble, and every spirit shall faint, and all knees shall be weak as water: behold, it cometh, and shall be brought to pass, saith the Lord GOD.',
 'metadata': {'chunk_id': 16777,
  'testament': 'The Old Testament of the King James Version of the Bible',
  'book': 'The Book of the Prophet Ezekiel',
  'beg_verse': '21:7',
  'end_verse': '21:7'}}

In [14]:
whoosh_idx = index.open_dir("../data/processed/bm25")

In [23]:
parser = QueryParser("text", whoosh_idx.schema, group=OrGroup)
parsed_query = parser.parse(query)

with whoosh_idx.searcher() as searcher:
    results = searcher.search(parsed_query, limit=50)
    w_indices = [hit['index'] for hit in results]
   


In [40]:
union = set(w_indices) | set(indices.squeeze().tolist())

In [42]:
len(union) 

98